In [1]:
import forecast_data, inspect
print(forecast_data.__file__)
print(hasattr(forecast_data, "TRADING_DAY_SPECS"), hasattr(forecast_data, "load_or_download_dukascopy"))

/Users/mihikaghaisas/Desktop/CMU/MLCapstone/Mihika/forecast_data.py
True True


# Path-Signature Volatility Forecasting Experiments

This notebook is intentionally **results-first**. It suppresses the long feature-generation logs and only displays compact experiment tables.

**Asset class is one setting:** set `ASSET` in section 2 to `"crypto"`, `"fx"` or `"equity"` and rerun the whole notebook. Every experiment, model and table below is unchanged; only the data source and the window lengths switch.

| ASSET | Intraday (5 min / 1 hour) | Daily horizons | Daily windows |
|---|---|---|---|
| crypto | Binance trades (BTCUSDT) | Yahoo BTC-USD | 1 / 7 / 30 days |
| fx | Dukascopy bid/ask ticks (EURUSD), free, no key | Yahoo EURUSD=X | 1 / 5 / 22 trading days |
| equity | none yet (needs a tick source) | Yahoo SPY (split-adjusted) | 1 / 5 / 22 trading days |

The experiment sequence is:

1. **Depth 3, full path:** HAR-RV-L, XGBoost (stats), Signature Ridge, XGBoost (signature).
2. **Depth 3 vs depth 4:** compare the two signature models. (Depth 4 is left disabled -- it crashes on this machine.)
3. **No augmentation vs time vs lead-lag + time:** compare the two signature models.
4. **Path dimensions:** compare price, price + activity, and the full source-compatible path.
5. **Full model comparison (daily horizons only):** HAR-RV-L, GARCH(1,1), XGBoost (stats/signature), Signature Ridge, MLP (stats/stats+signature), and a signature-LSTM, evaluated with a walk-forward backtest and ranked by RMSE with a Diebold-Mariano test against the best model. EGARCH is temporarily disabled.

Intraday paths are **price + activity + flow**. Binance flow is the signed taker volume; Dukascopy FX has no trade direction, so flow uses the tick rule (sign of each mid-price change). Daily Yahoo paths use **price + activity + range**, or **price + range** for FX (Yahoo has no FX volume).

**Baseline change:** the old "Random Forest on stats" baseline is now XGBoost on the same stats features, so the *only* difference between the "stats" and "signature" baselines is the feature set, not the model family.

**HAR change:** HAR follows the standard HAR-RV formulation -- rolling means of a daily variance proxy over the HAR windows, fit in log-variance space with a leverage term (HAR-RV-L), with a direct per-horizon regression.

**New dependencies:** `pip install arch torch statsmodels` (on top of the existing requirements) before running this notebook.


## 1. Imports and quiet output

In [2]:
from pathlib import Path
from dataclasses import replace
from collections import OrderedDict
import gc
import logging
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from forecast_data import (
    SPECS,
    TRADING_DAY_SPECS,
    build_dataset,
    ticks_to_bars,
    chronological_split,
    duration_steps,
    load_or_download_data,
)

from models_and_metrics import (
    EPS,
    WalkForwardResult,
    compare_models,
    evaluate_garch,
    evaluate_har_log,
    evaluate_model,
    fit_and_evaluate_standard_models,
    fit_mlp,
    fit_signature_lstm,
    fit_signature_ridge,
    fit_xgb,
    walk_forward_folds,
)

# Keep notebook output focused on final result tables.
# Change WARNING -> INFO if you want detailed progress.
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("forecast_data").setLevel(logging.WARNING)
logging.getLogger("models_and_metrics").setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## 2. Configuration
Change the asset class, dates and experiment settings here.

**Equity intraday:** no free tick source is wired in yet, so `"equity"` runs the daily horizons only. Options: WRDS TAQ (trades with exchange timestamps, the standard for equity microstructure work), or Alpaca's free plan (historical trades and minute bars since 2016, free API key). Either one only needs a `download_day` function in `forecast_data.py`, like the Binance and Dukascopy loaders.

In [3]:
DATA_ROOT = Path("data")

# =================================================
# ASSET CLASS -- the only line to change to rerun every experiment on another market
# =================================================
ASSET = "fx"   # "crypto" | "fx" | "equity"

# intraday: (tick source, symbol, start, end inclusive) -- used for 5 min / 1 hour.
# daily:    (Yahoo ticker, start, end inclusive)       -- used for the daily horizons.
ASSETS = {
    "crypto": {
        "intraday": ("binance", "BTCUSDT", "2026-08-01", "2026-08-31"),
        "daily": ("BTC-USD", "2020-01-01", "2026-09-16"),
    },
    "fx": {
        # Dukascopy: free bid/ask ticks, no API key. Weekends are skipped automatically.
        "intraday": ("dukascopy", "EURUSD", "2026-08-03", "2026-08-14"),
        "daily": ("EURUSD=X", "2012-01-01", "2026-11-11"),
    },
    "equity": {
        "intraday": None,  # see note above
        "daily": ("SPY", "2012-01-01", "2026-11-11"),
    },
}
INTRADAY = ASSETS[ASSET]["intraday"]
DAILY_TICKER, DAILY_START, DAILY_END = ASSETS[ASSET]["daily"]

# Crypto trades 7 days a week; equities/FX ~5. Windows are counted in bars, so
# their daily horizons use 1/5/22-bar windows instead of crypto's 1/7/30.
DAILY_SPECS = {h: SPECS[h] for h in ("1d", "7d", "30d")} if ASSET == "crypto" else TRADING_DAY_SPECS

# Yahoo has no FX volume, so the 'activity' path dimension would be a flat line.
HAS_DAILY_VOLUME = ASSET != "fx"

OUTPUT_ROOT = Path("outputs") / ASSET
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Existing cached files are reused unless this is True.
FORCE_DOWNLOAD = False

# Final chronological holdout for Experiments 1-4 (the signature ablations).
TEST_FRACTION = 0.20

# Main metric shown in the compact ablation tables. Experiment 5 additionally
# reports Diebold-Mariano significance against the best model per horizon.
PRIMARY_METRIC = "RMSE"

INTRADAY_HORIZONS = ["5m", "1h"] if INTRADAY else []

# Experiment 5 (full model comparison incl. GARCH/MLP/LSTM) only makes sense
# for the daily-bar horizons -- GARCH and the 5-day-window LSTM are built
# around daily variance, not 5s/1min bars.
DAILY_HORIZONS = list(DAILY_SPECS)
HORIZONS = INTRADAY_HORIZONS + DAILY_HORIZONS

HORIZON_LABELS = {
    "5m": "5 min",
    "1h": "1 hour",
    "1d": "1 day",
    "5d": "5 day",
    "7d": "7 day",
    "22d": "22 day",
    "30d": "30 day",
}

# Keep XGBoost enabled in the signature ablations (Experiments 2-4).
INCLUDE_XGB = True

# Walk-forward folds for Experiment 5 (expanding window, via TimeSeriesSplit).
N_SPLITS = 4

# MLP weight decay per horizon -- heavier regularization for the longer
# horizons, since fewer independent samples are available there.
WEIGHT_DECAY = {"1d": 1e-4, "5d": 5e-4, "7d": 5e-4, "22d": 1e-3, "30d": 1e-3}

# Memory: at most this many raw-tick / bars / feature-dataset objects are
# kept alive at once (LRU -- oldest evicted first).
MAX_CACHED_OBJECTS = 1


## 3. Helper functions

These helpers:

1. pick the right spec and data source for each horizon from `ASSET`,
2. load each raw source only once and cache each unique forecast-dataset specification, and
3. return compact result DataFrames instead of printing feature-generation details.

In [4]:
class BoundedCache(OrderedDict):
    """
    Small LRU cache so we never hold more than `max_size` big objects (raw
    ticks / bars / feature datasets) in memory at once. get_or_build reuses a
    cached value if present, otherwise builds it, stores it, and evicts the
    least-recently-used entry once the cache is over capacity.
    """
    def __init__(self, max_size=MAX_CACHED_OBJECTS):
        super().__init__()
        self.max_size = max_size

    def get_or_build(self, key, builder):
        if key in self:
            self.move_to_end(key)
            return self[key]
        value = builder()
        self[key] = value
        self.move_to_end(key)
        while len(self) > self.max_size:
            self.popitem(last=False)
            gc.collect()
        return value


RAW_CACHE = BoundedCache()
BARS_CACHE = BoundedCache()
DATASET_CACHE = BoundedCache()


def is_intraday(horizon):
    return horizon in INTRADAY_HORIZONS


def spec_for(horizon):
    """Default spec for a horizon under the current ASSET."""
    if is_intraday(horizon):
        return SPECS[horizon]
    spec = DAILY_SPECS[horizon]
    return spec if HAS_DAILY_VOLUME else replace(spec, dims=("price", "range"))


def get_raw_data(horizon):
    if is_intraday(horizon):
        source, symbol, start, end = INTRADAY
        load = lambda: load_or_download_data(source, start, end, symbol=symbol,
                                             data_root=DATA_ROOT, force_download=FORCE_DOWNLOAD)
        return RAW_CACHE.get_or_build(INTRADAY, load)

    key = ("yahoo", DAILY_TICKER, DAILY_START, DAILY_END)
    load = lambda: load_or_download_data("yahoo", DAILY_START, DAILY_END, ticker=DAILY_TICKER,
                                         data_root=DATA_ROOT, force_download=FORCE_DOWNLOAD)
    return RAW_CACHE.get_or_build(key, load)


def get_bars(horizon):
    def build():
        raw = get_raw_data(horizon)
        if not is_intraday(horizon):
            return raw.copy()
        # FX/equity ticks stop on weekends/overnight: drop bars in any 1h
        # stretch with no ticks. Crypto is 24/7, so nothing is dropped there.
        drop_gaps = None if INTRADAY[0] == "binance" else "1h"
        bars = ticks_to_bars(raw, spec_for(horizon).bar_freq, drop_gaps=drop_gaps)
        # The raw tick table is much larger than the resampled bars.
        # Do not retain both while signature features/models are built.
        RAW_CACHE.clear()
        del raw
        gc.collect()
        return bars

    return BARS_CACHE.get_or_build((ASSET, horizon), build)


def get_returns(horizon):
    """Daily log-returns for GARCH/EGARCH -- reuses the same cached bars."""
    return np.log(get_bars(horizon)["close"]).diff().dropna()


def spec_key(horizon, spec):
    return (
        ASSET,
        horizon,
        spec.depth,
        spec.lead_lag,
        spec.time_aug,
        tuple(spec.dims),
        spec.path_points,
        spec.sample_stride,
        spec.variance_method,
        spec.har_windows,
    )


def get_dataset(horizon, spec=None, include_lstm_seq=False):
    """
    One dataset object per (horizon, spec, include_lstm_seq) -- HAR/stats/
    signature features are always included; the LSTM's sub-window signature
    sequence (X_seq) is only computed when include_lstm_seq=True, so
    ablation experiments that don't need it never pay for it or hold it.
    """
    spec = spec or spec_for(horizon)
    key = spec_key(horizon, spec) + (include_lstm_seq,)
    return DATASET_CACHE.get_or_build(
        key, lambda: build_dataset(get_bars(horizon), spec, include_lstm_seq=include_lstm_seq)
    )


def signature_model_results(dataset):
    train_idx, test_idx = chronological_split(dataset, test_fraction=TEST_FRACTION)
    y_train, y_test = dataset.y[train_idx], dataset.y[test_idx]

    results = {}
    ridge = fit_signature_ridge(dataset.X_sig[train_idx], y_train)
    results["Signature Ridge"] = evaluate_model("Signature Ridge", ridge, dataset.X_sig[test_idx], y_test)

    if INCLUDE_XGB:
        xgb = fit_xgb(dataset.X_sig[train_idx], y_train)
        results["Signature XGBoost"] = evaluate_model("Signature XGBoost", xgb, dataset.X_sig[test_idx], y_test)

    return results


def result_rows(results, horizon, experiment, setting):
    return [
        {
            "Horizon": HORIZON_LABELS[horizon],
            "HorizonKey": horizon,
            "Experiment": experiment,
            "Setting": setting,
            "Model": model_name,
            **result.metrics,
        }
        for model_name, result in results.items()
    ]


def ordered_results(df):
    out = df.copy()
    out["Horizon"] = pd.Categorical(
        out["Horizon"],
        categories=[HORIZON_LABELS[h] for h in HORIZONS],
        ordered=True,
    )
    return out.sort_values(["Horizon", "RMSE"])


def show_baseline_table(df, metric=PRIMARY_METRIC):
    table = (
        df.pivot(index="Horizon", columns="Model", values=metric)
        .reindex([HORIZON_LABELS[h] for h in HORIZONS])
    )
    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"{ASSET} baseline -- {metric} (bold = lowest within horizon)")
    )
    return table


def show_ablation_table(df, metric=PRIMARY_METRIC):
    table = df.pivot_table(index=["Horizon", "Model"], columns="Setting", values=metric, aggfunc="first")

    order_map = {HORIZON_LABELS[h]: i for i, h in enumerate(HORIZONS)}
    temp = table.reset_index()
    temp["_order"] = temp["Horizon"].map(order_map)
    temp = temp.sort_values(["_order", "Model"]).drop(columns="_order")
    table = temp.set_index(["Horizon", "Model"])

    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"{ASSET} {metric} comparison (bold = lowest across settings in each row)")
    )
    return table


def best_dims_for_horizon(horizon, dimension_df):
    """Read Experiment 4's own ablation results and return the path-dimension
    tuple with the lowest RMSE for this horizon -- reused directly instead of
    re-running the dimension search inside Experiment 5."""
    sub = dimension_df[dimension_df["HorizonKey"] == horizon]
    best_setting = sub.loc[sub["RMSE"].idxmin(), "Setting"]
    return dimension_variants(horizon)[best_setting]


## 4. Experiment 1 — Baseline: HAR-RV-L, XGBoost (stats), Signature Ridge, XGBoost (signature)

This is the main baseline.

For each horizon:

- **HAR-RV-L** uses log-RV over the three HAR windows + a leverage term (see the intro cell).
- **XGBoost (stats)** uses the summary-statistic features -- same model family as the signature baseline below, for an apples-to-apples comparison.
- **Signature Ridge** uses depth-3 signatures.
- **XGBoost (signature)** uses depth-3 signatures.

The baseline signature path uses both **lead-lag and time augmentation**, with the full path for the asset (see the intro cell).


In [5]:
baseline_rows = []

for horizon in HORIZONS:
    spec = replace(spec_for(horizon), depth=3, lead_lag=True, time_aug=True)

    dataset = get_dataset(horizon, spec)
    train_idx, test_idx = chronological_split(dataset, test_fraction=TEST_FRACTION)
    horizon_steps = duration_steps(spec.horizon, spec.bar_freq)

    results = fit_and_evaluate_standard_models(dataset, train_idx, test_idx, horizon_steps)
    baseline_rows.extend(
        result_rows(results, horizon=horizon, experiment="Baseline",
                    setting="Depth 3 | full path | lead-lag + time")
    )

    # Only the compact metric rows are needed later. Keeping datasets and
    # fitted model objects across horizons is what caused the kernel crash.
    del results, dataset, train_idx, test_idx
    DATASET_CACHE.clear()
    gc.collect()

baseline_df = ordered_results(pd.DataFrame(baseline_rows))
baseline_table = show_baseline_table(baseline_df)


[6/12] Skipping 2026-08-08: Saturday, FX market closed
[6/12] Skipping 2026-08-08: Saturday, FX market closed


Model,HAR-RV-L,Signature Ridge,XGBoost (signature),XGBoost (stats)
Horizon,,,,
5 min,7.16264e-05,5.05102e-05,6.56157e-05,6.38727e-05
1 hour,0.000165863,0.000234083,0.000205801,0.000205801
1 day,0.00183395,0.0017894,0.00178817,0.00180829
5 day,0.00281804,0.002866,0.00283053,0.00278013
22 day,0.00480846,0.00439692,0.00448499,0.00425882


## 5. Experiment 2 — Depth 3 vs depth 4

Everything except signature depth is held fixed:

- full source-compatible path dimensions
- lead-lag augmentation
- time augmentation

Only the two signature models are shown because HAR and XGBoost (stats) are unaffected by signature depth.

In [6]:
# depth_rows = []

# for horizon in HORIZONS:
#     for depth in [3, 4]:
#         spec = replace(spec_for(horizon), depth=depth, lead_lag=True, time_aug=True)
#         dataset = get_dataset(horizon, spec)
#         results = signature_model_results(dataset)
#         depth_rows.extend(result_rows(results, horizon=horizon, experiment="Depth", setting=f"Depth {depth}"))

# depth_df = ordered_results(pd.DataFrame(depth_rows))
# depth_table = show_ablation_table(depth_df)


## 6. Experiment 3 — Augmentation

Depth is fixed at 3 and the full source-compatible dimensions are used.

The three paths are:

1. **None** — raw path only
2. **Time** — time augmentation only
3. **Lead-lag + time** — lead-lag transform followed by time augmentation

In [7]:
AUGMENTATIONS = {
    "None": {"lead_lag": False, "time_aug": False},
    "Time": {"lead_lag": False, "time_aug": True},
    "Lead-lag + time": {"lead_lag": True, "time_aug": True},
}

augmentation_rows = []

for horizon in HORIZONS:
    for setting, kwargs in AUGMENTATIONS.items():
        spec = replace(spec_for(horizon), depth=3, **kwargs)
        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)
        augmentation_rows.extend(
            result_rows(results, horizon=horizon, experiment="Augmentation", setting=setting)
        )

augmentation_df = ordered_results(pd.DataFrame(augmentation_rows))
augmentation_table = show_ablation_table(augmentation_df)


[6/12] Skipping 2026-08-08: Saturday, FX market closed
[6/12] Skipping 2026-08-08: Saturday, FX market closed


## 7. Experiment 4 — Path dimensions

Depth is fixed at 3 and **lead-lag + time augmentation** is used throughout.

### Intraday (tick data): 5 min and 1 hour
- Price
- Price + activity
- Price + activity + flow

### Daily (Yahoo)
Yahoo OHLCV has no signed trade flow, so the third dimension is the daily range path:

- Price
- Price + activity
- Price + activity + range

For **FX**, Yahoo has no volume, so the daily variants are **Price** and **Price + range**.

In [8]:
def dimension_variants(horizon):
    if is_intraday(horizon):
        return {
            "Price": ("price",),
            "Price + activity": ("price", "activity"),
            "Price + activity + flow": ("price", "activity", "flow"),
        }
    if not HAS_DAILY_VOLUME:
        return {
            "Price": ("price",),
            "Price + range": ("price", "range"),
        }
    return {
        "Price": ("price",),
        "Price + activity": ("price", "activity"),
        "Price + activity + range": ("price", "activity", "range"),
    }


dimension_rows = []

for horizon in HORIZONS:
    for setting, dims in dimension_variants(horizon).items():
        spec = replace(spec_for(horizon), depth=3, lead_lag=True, time_aug=True, dims=dims)
        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)
        dimension_rows.extend(
            result_rows(results, horizon=horizon, experiment="Dimensions", setting=setting)
        )

dimension_df = ordered_results(pd.DataFrame(dimension_rows))
dimension_table = show_ablation_table(dimension_df)


[6/12] Skipping 2026-08-08: Saturday, FX market closed
[6/12] Skipping 2026-08-08: Saturday, FX market closed


## 8. Experiment 5 -- full model comparison (daily horizons)

Only the daily-bar horizons run here -- GARCH and the 5-day-window signature-LSTM are built around a daily variance proxy, not 5s/1min bars. EGARCH is temporarily disabled.

For each horizon:

- Path dimensions are fixed to whichever setting had the lowest RMSE for that horizon in Experiment 4 above.
- Every model is evaluated on the same **walk-forward (expanding-window) backtest** (`N_SPLITS` folds), not a single train/test split.
- Models: **HAR-RV-L**, **GARCH(1,1)**, **XGBoost (stats)**, **XGBoost (signature)**, **Signature Ridge**, **MLP (stats)**, **MLP (stats + signature)**, **Signature LSTM**.
- The table is ranked by RMSE, with a **Diebold-Mariano test** of every model against the current best -- a low p-value means that model is *significantly* worse than the best, not just numerically worse.

**Memory:** each horizon's dataset is cleared before moving to the next horizon.


In [9]:
def run_full_comparison(horizon, dimension_df, n_splits=N_SPLITS):
    spec = replace(spec_for(horizon), depth=3, lead_lag=True, time_aug=True,
                   dims=best_dims_for_horizon(horizon, dimension_df))
    # include_lstm_seq=True computes X_har/X_stats/X_sig/X_seq together in one pass.
    dataset = get_dataset(horizon, spec, include_lstm_seq=True)
    horizon_steps = duration_steps(spec.horizon, spec.bar_freq)
    returns = get_returns(horizon)
    weight_decay = WEIGHT_DECAY[horizon]

    model_names = [
        "HAR-RV-L", "GARCH(1,1)", "XGBoost (stats)",
        "XGBoost (signature)", "Signature Ridge", "MLP (stats)",
        "MLP (stats+signature)", "Signature LSTM",
    ]
    preds = {name: [] for name in model_names}
    truths = []

    for train_idx, test_idx in walk_forward_folds(len(dataset.y), n_splits=n_splits):
        y_train, y_test = dataset.y[train_idx], dataset.y[test_idx]
        truths.append(y_test)

        har = evaluate_har_log(
            dataset.X_har[train_idx], dataset.y_har_log[train_idx],
            dataset.X_har[test_idx], y_test, horizon_steps,
        )
        preds["HAR-RV-L"].append(har.predictions)

        returns_train = returns.loc[:dataset.times[train_idx[-1]]]
        anchor_times = pd.DatetimeIndex(dataset.times[test_idx])
        g = evaluate_garch(returns_train, returns, anchor_times, y_test, horizon_steps, vol="GARCH")
        preds["GARCH(1,1)"].append(g.predictions)
        # EGARCH temporarily disabled:
        # eg = evaluate_garch(returns_train, returns, anchor_times, y_test, horizon_steps, vol="EGARCH")
        # preds["EGARCH(1,1)"].append(eg.predictions)

        xgb_stats = fit_xgb(dataset.X_stats[train_idx], y_train)
        preds["XGBoost (stats)"].append(np.maximum(xgb_stats.predict(dataset.X_stats[test_idx]), EPS))

        xgb_sig = fit_xgb(dataset.X_sig[train_idx], y_train)
        preds["XGBoost (signature)"].append(np.maximum(xgb_sig.predict(dataset.X_sig[test_idx]), EPS))

        ridge = fit_signature_ridge(dataset.X_sig[train_idx], y_train)
        preds["Signature Ridge"].append(np.maximum(ridge.predict(dataset.X_sig[test_idx]), EPS))

        mlp = fit_mlp(dataset.X_stats[train_idx], y_train, weight_decay=weight_decay)
        preds["MLP (stats)"].append(np.maximum(mlp.predict(dataset.X_stats[test_idx]), EPS))

        X_stats_sig_train = np.hstack([dataset.X_stats[train_idx], dataset.X_sig[train_idx]])
        X_stats_sig_test = np.hstack([dataset.X_stats[test_idx], dataset.X_sig[test_idx]])
        mlp_sig = fit_mlp(X_stats_sig_train, y_train, weight_decay=weight_decay)
        preds["MLP (stats+signature)"].append(np.maximum(mlp_sig.predict(X_stats_sig_test), EPS))

        lstm = fit_signature_lstm(dataset.X_seq[train_idx], y_train, weight_decay=weight_decay)
        preds["Signature LSTM"].append(np.maximum(lstm.predict(dataset.X_seq[test_idx]), EPS))

    y_true = np.concatenate(truths)
    results = {name: WalkForwardResult(name, y_true, np.concatenate(p)) for name, p in preds.items()}
    return compare_models(results, horizon_steps)


full_comparison_tables = {}
for horizon in DAILY_HORIZONS:
    table = run_full_comparison(horizon, dimension_df)
    full_comparison_tables[horizon] = table
    display(
        table.style
        .format({"RMSE": "{:.6g}", "DM stat vs best": "{:.3f}", "DM p-value": "{:.3f}"})
        .apply(lambda col: ["font-weight: bold" if b else "" for b in table["Best"]], subset=["Model", "RMSE"])
        .set_caption(f"{ASSET} {HORIZON_LABELS[horizon]} -- best model is bolded; "
                     "DM p-value < 0.05 means significantly worse than best")
    )

    # Drop this horizon's dataset before building the next one.
    DATASET_CACHE.clear()
    gc.collect()


,Model,RMSE,Best,DM stat vs best,DM p-value
0,Signature Ridge,0.00206529,True,nan,nan
1,HAR-RV-L,0.00209442,False,1.653,0.098
2,"GARCH(1,1)",0.00212022,False,3.403,0.001
3,XGBoost (signature),0.00213042,False,3.766,0.000
4,XGBoost (stats),0.0021508,False,4.483,0.000
5,Signature LSTM,0.00227224,False,5.678,0.000
6,MLP (stats),0.00234201,False,13.117,0.000
7,MLP (stats+signature),0.00235494,False,13.670,0.000


,Model,RMSE,Best,DM stat vs best,DM p-value
0,HAR-RV-L,0.00317471,True,nan,nan
1,"GARCH(1,1)",0.00319526,False,0.266,0.790
2,XGBoost (signature),0.00327711,False,1.916,0.055
3,XGBoost (stats),0.00329731,False,2.219,0.026
4,Signature LSTM,0.00334733,False,3.351,0.001
5,Signature Ridge,0.00353829,False,2.503,0.012
6,MLP (stats),0.00414638,False,12.083,0.000
7,MLP (stats+signature),0.00417476,False,12.078,0.000


,Model,RMSE,Best,DM stat vs best,DM p-value
0,"GARCH(1,1)",0.00519185,True,nan,nan
1,Signature LSTM,0.00553823,False,1.542,0.123
2,HAR-RV-L,0.00554023,False,1.355,0.175
3,XGBoost (signature),0.00557932,False,1.317,0.188
4,XGBoost (stats),0.00578958,False,1.767,0.077
5,Signature Ridge,0.0066015,False,2.014,0.044
6,MLP (stats+signature),0.00749997,False,5.039,0.000
7,MLP (stats),0.00752948,False,5.057,0.000


## 9. Combined results

Combines the signature-ablation experiments (1-4) into one tidy DataFrame and shows the top rows per horizon. Experiment 5's per-horizon tables above are the "which model wins" answer.


In [10]:
all_results = ordered_results(
    pd.concat([baseline_df, augmentation_df, dimension_df], ignore_index=True)  # add depth_df if re-enabled
)

for horizon in HORIZONS:
    display(
        all_results[all_results["HorizonKey"] == horizon][["Horizon", "Experiment", "Setting", "Model", "RMSE"]]
        .head(5)
        .style.format({"RMSE": "{:.6g}"})
        .set_caption(f"{ASSET} {HORIZON_LABELS[horizon]} -- top 5")
    )


,Horizon,Experiment,Setting,Model,RMSE
0,5 min,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,5.05102e-05
20,5 min,Augmentation,Lead-lag + time,Signature Ridge,5.05102e-05
50,5 min,Dimensions,Price + activity + flow,Signature Ridge,5.05102e-05
51,5 min,Dimensions,Price + activity,Signature Ridge,5.34164e-05
21,5 min,Augmentation,Time,Signature Ridge,5.97955e-05


,Horizon,Experiment,Setting,Model,RMSE
26,1 hour,Augmentation,None,Signature Ridge,0.000152199
4,1 hour,Baseline,Depth 3 | full path | lead-lag + time,HAR-RV-L,0.000165863
56,1 hour,Dimensions,Price + activity,Signature Ridge,0.000178579
57,1 hour,Dimensions,Price,Signature Ridge,0.000193101
5,1 hour,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (stats),0.000205801


,Horizon,Experiment,Setting,Model,RMSE
32,1 day,Augmentation,Time,Signature Ridge,0.00178596
33,1 day,Augmentation,Time,Signature XGBoost,0.00178672
8,1 day,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (signature),0.00178817
34,1 day,Augmentation,Lead-lag + time,Signature XGBoost,0.00178817
62,1 day,Dimensions,Price + range,Signature XGBoost,0.00178817


,Horizon,Experiment,Setting,Model,RMSE
12,5 day,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (stats),0.00278013
38,5 day,Augmentation,Time,Signature XGBoost,0.00280781
13,5 day,Baseline,Depth 3 | full path | lead-lag + time,HAR-RV-L,0.00281804
14,5 day,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (signature),0.00283053
39,5 day,Augmentation,Lead-lag + time,Signature XGBoost,0.00283053


,Horizon,Experiment,Setting,Model,RMSE
16,22 day,Baseline,Depth 3 | full path | lead-lag + time,XGBoost (stats),0.00425882
44,22 day,Augmentation,None,Signature Ridge,0.00438948
17,22 day,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.00439692
45,22 day,Augmentation,Lead-lag + time,Signature Ridge,0.00439692
70,22 day,Dimensions,Price + range,Signature Ridge,0.00439692


## 10. Save the result tables

Saves the compact experiment results and Experiment 5's model-comparison tables to `outputs/<ASSET>/`, so runs for different asset classes never overwrite each other.


In [11]:
baseline_df.to_csv(OUTPUT_ROOT / "baseline_results.csv", index=False)
augmentation_df.to_csv(OUTPUT_ROOT / "augmentation_ablation_results.csv", index=False)
dimension_df.to_csv(OUTPUT_ROOT / "dimension_ablation_results.csv", index=False)
all_results.to_csv(OUTPUT_ROOT / "all_experiment_results.csv", index=False)

for horizon, table in full_comparison_tables.items():
    table.to_csv(OUTPUT_ROOT / f"full_comparison_{horizon}.csv", index=False)

print("Saved compact result CSVs to:", OUTPUT_ROOT.resolve())


Saved compact result CSVs to: /Users/mihikaghaisas/Desktop/CMU/MLCapstone/Mihika/outputs/fx
